# Identification of Spatial Selectivity–Associated Genes

To identify genes associated with spatial selectivity, we employed a multi-step analytical pipeline integrating single-cell–level screening and mouse-level consistency assessment.

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib as mpl
from datetime import date
import scanpy as sc
import omicverse as ov
from matplotlib_venn import venn3
import matplotlib.pyplot as plt
from pathlib import Path

import pickle as pkl

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from pydeseq2.utils import load_example_data

SAVE = False  # whether to save the outputs of this notebook

In [ ]:
mpl.rcParams["font.family"] = "Arial"          
mpl.rcParams["font.sans-serif"] = ["Arial"]    
mpl.rcParams["pdf.fonttype"] = 42              
mpl.rcParams["ps.fonttype"]  = 42
mpl.rcParams["savefig.dpi"] = 300

mpl.rcParams["figure.dpi"]  = 300

In [ ]:
input_path = r"F:\Data\snRNAseq\PlaceCell\Results\DEGs\FindMarkers_wilcoxon"
output_path = Path(r"F:\Data\snRNAseq\PlaceCell\Results\Venn\abslogfc0.1")
output_path.mkdir(parents=True, exist_ok=True)
today = date.today().strftime("%Y%m%d")

df_ns = pd.read_csv(os.path.join(input_path,"nonplace_silent_deg_wilcoxon_20260129.csv"),index_col=0)
df_ps = pd.read_csv(os.path.join(input_path,"place_silent_deg_wilcoxon_20260129.csv"),index_col=0)
df_pn = pd.read_csv(os.path.join(input_path,"place_nonplace_deg_wilcoxon_20260129.csv"),index_col=0)

---

### 1. Exploratory DEG Screening at the Single-Cell Level

As an initial exploratory step, differential expression analysis was performed by treating individual cells as independent samples. Differentially expressed genes (DEGs) were identified using relaxed criteria (|log2FC| > 0.1 and unadjusted P value < 0.05). These permissive thresholds were intentionally applied to retain a broad pool of candidate genes for subsequent filtering and robustness analyses.

##### Definition of spatial selectivity–associated genes

Spatial selectivity–associated genes were defined based on logical set operations across pairwise condition comparisons:
- Genes that were differentially expressed in both *place vs. non-place* and *place vs. silent* comparisons,
- Excluding genes detected exclusively in the *non-place vs. silent* comparison.

This definition aimed to prioritize genes specifically associated with place cell identity rather than differences attributable solely to neuronal activity.

##### Definition of activity-associated genes

Activity-associated genes were defined using complementary logical criteria:
- Genes that were differentially expressed in both *place vs. silent* and *non-place vs. silent* comparisons,
- Excluding genes identified exclusively in the *place vs. non-place* comparison.

This definition enriched for genes related to general neuronal activity independent of spatial selectivity.

---


In [ ]:
df_ns

In [ ]:
# Global DEG filtering parameters
P_COL  = "p_val"
LFC_COL = "avg_log2FC"

P_TH  = 0.05
LFC_TH = 0.1

In [ ]:
def filter_deg(
    df,
    direction="all",      # "all" | "up" | "down"
    p_th=P_TH,
    lfc_th=LFC_TH,
    p_col=P_COL,
    lfc_col=LFC_COL
):
    """
    Filter DEG table by p-value and direction.

    Parameters
    ----------
    direction : {"all", "up", "down"}
        all  : p < p_th
        up   : p < p_th & logFC > lfc_th
        down : p < p_th & logFC < -lfc_th
    """

    if direction == "all":
        q = f"{p_col} < @p_th and abs({lfc_col}) > @lfc_th"
    elif direction == "up":
        q = f"{p_col} < @p_th and {lfc_col} > @lfc_th"
    elif direction == "down":
        q = f"{p_col} < @p_th and {lfc_col} < -@lfc_th"
    else:
        raise ValueError("direction must be one of {'all','up','down'}")

    return df.query(q).copy()


In [ ]:
deg_ps = filter_deg(df_ps, direction="all")
deg_ns = filter_deg(df_ns, direction="all")
deg_pn = filter_deg(df_pn, direction="all")

genes_ps = set(deg_ps.index)
genes_ns = set(deg_ns.index)
genes_pn = set(deg_pn.index)


In [ ]:
plt.figure(figsize=(5,5))
venn_sets = {
    "Place vs NonPlace": genes_pn,
    "Place vs Silent": genes_ps,
    "NonPlace vs Silent": genes_ns,
}

venn3(
    venn_sets.values(),
    set_labels=venn_sets.keys()
)

plt.title("DEGs Overlap (All)")
plt.tight_layout()
plt.savefig(os.path.join(output_path, f"Venn_all_lfc{LFC_TH}_{today}.png"), dpi = 300)
plt.show()


In [ ]:
deg_ps_up = filter_deg(df_ps, direction="up")
deg_ns_up = filter_deg(df_ns, direction="up")
deg_pn_up = filter_deg(df_pn, direction="up")

genes_ps_up = set(deg_ps_up.index)
genes_ns_up = set(deg_ns_up.index)
genes_pn_up = set(deg_pn_up.index)


In [ ]:
plt.figure(figsize=(5,5))
venn_sets = {
    "Place vs NonPlace": genes_pn_up,
    "Place vs Silent": genes_ps_up,
    "NonPlace vs Silent": genes_ns_up,
}

venn3(
    venn_sets.values(),
    set_labels=venn_sets.keys()
)

plt.title("DEGs Overlap (Upregulated)")
plt.tight_layout()
plt.savefig(os.path.join(output_path, f"Venn_up_lfc{LFC_TH}_{today}.png"), dpi = 300)
plt.show()


In [ ]:
deg_ps_down = filter_deg(df_ps, direction="down")
deg_ns_down = filter_deg(df_ns, direction="down")
deg_pn_down = filter_deg(df_pn, direction="down")

genes_ps_down = set(deg_ps_down.index)
genes_ns_down = set(deg_ns_down.index)
genes_pn_down = set(deg_pn_down.index)


In [ ]:
plt.figure(figsize=(5,5))
venn_sets = {
    "Place vs NonPlace": genes_pn_down,
    "Place vs Silent": genes_ps_down,
    "NonPlace vs Silent": genes_ns_down,
}

venn3(
    venn_sets.values(),
    set_labels=venn_sets.keys()
)

plt.title("DEGs Overlap (Downregulated)")
plt.tight_layout()
plt.savefig(os.path.join(output_path, f"Venn_down_lfc{LFC_COL}_{today}.png"), dpi = 300)
plt.show()


In [ ]:
# Spatial selectivity–associated genes:
spatial_candidate_genes_all = (genes_pn & genes_ps) - genes_ns

# Activity-associated genes:
activity_candidate_genes_all = (genes_ns & genes_ps) - genes_pn

print(f"All spatial candidate genes:  {len(spatial_candidate_genes_all)}")
print(f"All activity candidate genes: {len(activity_candidate_genes_all)}")

activity_candidate_df = pd.DataFrame(
    sorted(activity_candidate_genes_all),
    columns=["gene"]
)

activity_candidate_df.to_csv(
    os.path.join(
        output_path,
        f"activity_candidate_genes_all_lfc{LFC_TH}_{today}.csv"
    ),
    index=False
)

spatial_candidate_df = pd.DataFrame(
    sorted(spatial_candidate_genes_all),
    columns=["gene"]
)

spatial_candidate_df.to_csv(
    os.path.join(
        output_path,
        f"spatial_candidate_genes_all_lfc{LFC_TH}_{today}.csv"
    ),
    index=False
)


In [ ]:
# Upregulated spatial selectivity–associated genes:
# (place > non-place) AND (place > silent), but NOT (non-place > silent)
spatial_candidate_genes_up = (genes_pn_up & genes_ps_up) - genes_ns_up

# Upregulated activity-associated genes:
# (place > silent) AND (non-place > silent), but NOT (place > non-place)
activity_candidate_genes_up = (genes_ns_up & genes_ps_up) - genes_pn_up


print(f"Upregulated spatial candidate genes:  {len(spatial_candidate_genes_up)}")
print(f"Upregulated activity candidate genes: {len(activity_candidate_genes_up)}")

# Save upregulated activity-associated genes

activity_candidate_up_df = pd.DataFrame(
    sorted(activity_candidate_genes_up),
    columns=["gene"]
)

activity_candidate_up_df.to_csv(
    os.path.join(
        output_path,
        f"activity_candidate_genes_up_lfc{LFC_TH}_{today}.csv"
    ),
    index=False
)

# Save upregulated spatial selectivity–associated genes

spatial_candidate_up_df = pd.DataFrame(
    sorted(spatial_candidate_genes_up),
    columns=["gene"]
)

spatial_candidate_up_df.to_csv(
    os.path.join(
        output_path,
        f"spatial_candidate_genes_up_lfc{LFC_TH}_{today}.csv"
    ),
    index=False
)


In [ ]:
# ============================================================
# Downregulated candidate gene sets (strictly downregulated)
# ============================================================

# Downregulated spatial selectivity–associated genes:
# (place < non-place) AND (place < silent), but NOT (non-place < silent)
spatial_candidate_genes_down = (genes_pn_down & genes_ps_down) - genes_ns_down

# Downregulated activity-associated genes:
# (place < silent) AND (non-place < silent), but NOT (place < non-place)
activity_candidate_genes_down = (genes_ns_down & genes_ps_down) - genes_pn_down


print(f"Downregulated spatial candidate genes:  {len(spatial_candidate_genes_down)}")
print(f"Downregulated activity candidate genes: {len(activity_candidate_genes_down)}")


# ============================================================
# Save downregulated activity-associated genes
# ============================================================

activity_candidate_down_df = pd.DataFrame(
    sorted(activity_candidate_genes_down),
    columns=["gene"]
)

activity_candidate_down_df.to_csv(
    os.path.join(
        output_path,
        f"activity_candidate_genes_down_lfc{LFC_TH}_{today}.csv"
    ),
    index=False
)


# ============================================================
# Save downregulated spatial selectivity–associated genes
# ============================================================

spatial_candidate_down_df = pd.DataFrame(
    sorted(spatial_candidate_genes_down),
    columns=["gene"]
)

spatial_candidate_down_df.to_csv(
    os.path.join(
        output_path,
        f"spatial_candidate_genes_down_lfc{LFC_TH}_{today}.csv"
    ),
    index=False
)


In [ ]:
# up / down 不应有重叠
assert len(spatial_candidate_genes_up & spatial_candidate_genes_down) == 0
assert len(activity_candidate_genes_up & activity_candidate_genes_down) == 0

# direction 纯净性检查
assert len(spatial_candidate_genes_down & genes_ns_down) == 0
assert len(activity_candidate_genes_down & genes_pn_down) == 0


## 2. Mouse-level robustness and consistency analysis

To assess robustness across animals, we conducted a mouse-level consistency analysis. For each mouse, log2 fold changes were independently calculated for all pairwise condition comparisons (place, non-place, and silent), noting that each mouse contained only two of the three conditions.

For each gene, a consistency score was computed to quantify the reproducibility of expression trends across mice:

\[
\text{Consistency score} =
\frac{
\left(\frac{\text{replicated trends}}{N_{\text{place vs non-place}}}\right)
+
\left(\frac{\text{replicated trends}}{N_{\text{place vs silent}}}\right)
+
\left(\frac{\text{replicated trends}}{N_{\text{non-place vs silent}}}\right)
}{3}
\]

This metric captures the extent to which the direction of differential expression is preserved across animals, thereby prioritizing genes with stable and reproducible spatial selectivity–associated patterns.

In [ ]:
adata = sc.read_h5ad(r"F:\Data\snRNAseq\PlaceCell\FilteredData\PCAnalysis_all_QC_20260127.h5ad")
adata

In [ ]:
adata_pn = adata[adata.obs["sample"].isin(["PC94", "PC110", "PC112", "PC114"])].copy()   #place vs nonplace
adata_ps = adata[adata.obs["sample"].isin(["PC103", "PC106", "PC118"])].copy() #place vs silent
adata_ns = adata[adata.obs["sample"].isin(["PC92", "PC105", "PC132"])].copy()  #nonplace vs silent

adata_pn.obs["condition"].value_counts()
adata_ps.obs["condition"].value_counts()
adata_ns.obs["condition"].value_counts()

In [ ]:
adata_sub.obs["sample"].value_counts()

In [ ]:
samples = adata_sub.obs["sample"].unique()
conditions = adata_sub.obs["condition"].unique()

In [ ]:
pbs = []

for sample in samples:
    for condition in conditions:

        mask = (
            (adata_sub.obs["sample"] == sample) &
            (adata_sub.obs["condition"] == condition)
        )

        if mask.sum() == 0:
            raise ValueError(f"No cells for sample={sample}, condition={condition}")

        
        subset = adata_sub[mask].copy()
        subset.X = subset.layers["counts"]

        rep = sc.AnnData(
            X=subset.X.sum(axis=0, keepdims=True),
            var=subset.var
        )
        rep.obs_names = [f"{sample}_{condition}"]
        rep.obs["sample"] = sample
        rep.obs["condition"] = condition

        pbs.append(rep)


In [ ]:
pb = sc.concat(pbs)

In [ ]:
counts = pd.DataFrame(
    pb.X,
    index=pb.obs_names,   # ← 非常关键
    columns=pb.var_names
)
counts = counts.astype(int)
metadata = pb.obs[["sample", "condition"]]
assert (counts.index == metadata.index).all()


In [ ]:
((counts % 1) == 0).all().all()

In [ ]:
inference = DefaultInference(n_cpus=8)

log2fc_dict = {}   # key = mouse, value = pd.Series (genes)

for mouse in metadata["sample"].unique():

    # -------- 1. 拆分 counts / metadata（在建模之前！） --------
    mask = metadata["sample"] == mouse

    counts_m   = counts.loc[mask]        # (2 × n_genes)
    metadata_m = metadata.loc[mask]

    # 安全检查
    if counts_m.shape[0] != 2:
        print(f"Skip {mouse}: {counts_m.shape[0]} samples")
        continue

    # -------- 2. 构建 DeseqDataSet（只为 size factor） --------
    dds = DeseqDataSet(
        counts=counts_m,
        metadata=metadata_m,
        design_factors=["condition"],
        ref_level=["condition", "nonplace"],
        inference=inference,
    )

    dds.fit_size_factors()
    norm = dds.layers["normed_counts"]    # shape: (2, n_genes)

    # -------- 3. 明确行顺序，算 per-gene log2FC --------
    cond = metadata_m["condition"].values
    idx_place    = cond == "place"
    idx_nonplace = cond == "nonplace"

    log2fc = np.log2(
        (norm[idx_place, :].ravel() + 1) /
        (norm[idx_nonplace, :].ravel() + 1)
    )

    # -------- 4. 存成带基因名的 Series --------
    log2fc_dict[mouse] = pd.Series(
        log2fc,
        index=dds.var_names,
        name=mouse
    )

# -------- 5. 合并成 gene × mouse 的 DataFrame --------
log2fc_df = pd.concat(log2fc_dict, axis=1)

print(log2fc_df.shape)
# (n_genes, n_mice)


In [ ]:
subset = log2fc_df.reindex(spatial_candidate_genes_up)

consistent_up = (subset > 0).sum(axis=1) >= 3

In [ ]:
consistent_up = consistent_up[consistent_up == True]

In [ ]:
consistent_up.index

In [ ]:
consistent_up_df = pd.DataFrame(
    sorted(consistent_up.index),
    columns=["gene"]
)

consistent_up_df.to_csv(
    os.path.join(
        output_path,
        f"consistent_up_df_{today}.csv"
    ),
    index=False
)

## 3. Permutation-based validation (shuffle test)

Finally, a permutation-based shuffle test was applied to disrupt the correspondence between gene identities and expression values. This procedure was used as a negative control to estimate the likelihood that the observed consistency arose by chance, providing an additional layer of validation for the identified spatial selectivity–associated genes.

In [ ]:
candidate_genes = (
    pd.DataFrame(
        sorted(activity_candidate_genes_all),
        columns=["gene"]
    )
)

candidate_genes.to_csv(
    os.path.join(output_path,f"activity_candidate_genes_lfc{LFC_COL}_{today}.csv"),
    index=False
)

candidate_genes = (
    pd.DataFrame(
        sorted(spatial_candidate_genes_all),
        columns=["gene"]
    )
)

candidate_genes.to_csv(
    os.path.join(output_path,f"spatial_candidate_genes_lfc{LFC_COL}_{today}.csv"),
    index=False
)


In [ ]:

BASE_DIR = r"F:\Data\snRNAseq\PlaceCell\FilteredData"
INPUT_FILE = f"PCAnalysis_All_QC_20260127.h5ad"

today = date.today().strftime("%Y%m%d")
input_path = os.path.join(BASE_DIR, INPUT_FILE)

In [ ]:
adata = sc.read_h5ad(input_path)
adata

In [ ]:
mouse = "PC118"
adata_m = adata[adata.obs["sample"] == mouse].copy()

sc.pp.normalize_total(adata_m, target_sum=1e4)
sc.pp.log1p(adata_m)


In [ ]:
Xp = adata_m[adata_m.obs["condition"] == "place"].X
Xn = adata_m[adata_m.obs["condition"] == "nonplace"].X


In [ ]:
adata_m

In [ ]:
logFC = np.asarray(Xp.mean(axis=0) - Xn.mean(axis=0)).ravel()

logFC = pd.Series(
    logFC,
    index=adata_m.var_names,
    name="logFC"
)
logFC["Stmn4"]
logFC
